#**Packages**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#**Necessary Functions for Testing**

In [ ]:
def generate_random_ULTM(n, low, high):
    # Create an n x n matrix filled with random integers in the specified range
    matrix = np.random.randint(low, high + 1, size=(n, n))
    # Set upper triangular part (excluding the diagonal) to zero
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j] = 0
    # Set diagonal entries to 1
    for i in range(n):
        for j in range(n):
            if i==j:
                matrix[i][j] = 1
    return matrix


def generate_random_BULTM(n, low, high):
    # Create an n x n matrix filled with random integers in the specified range
    matrix = np.random.randint(low, high + 1, size=(n, n))
    # Set upper triangular part (excluding the diagonal) to 0
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j] = 0
    # Set lower triangular part (excluding the first two subdiagonals) to 0
    for i in range(n):
        for j in range(i-2):
            matrix[i][j] = 0
    # Set diagonal entries to 1
    for i in range(n):
        for j in range(n):
            if i==j:
                matrix[i][j] = 1
    return matrix


def generate_random_UTM(n, low, high):
    # Create an n x n matrix filled with random integers in the specified range
    U_matrix = np.random.randint(low, high + 1, size=(n, n))
    # Set lower triangular part (excluding the diagonal) to zero
    for i in range(n):
        for j in range(i):
            U_matrix[i][j] = 0
    return U_matrix


def construct_LU(L,U):
    # Combine L and U into a single 2D array
    if len(L) == len(U):
        n = len(L)
    else:
        print("L and U are different sizes")
    LU = [[0] * n for i in range(n)]
    for i in range(n):
        for j in range(n):
            if i > j:
                LU[i][j] = L[i][j]
            else:
                LU[i][j] = U[i][j]
    return LU


def generate_random_vector(n, low, high):
    # Generate a random n x 1 vector with integer entries between low and high
    vector = np.random.randint(low, high + 1, size=(n, 1))
    return vector


def vector_difference(true, computed):
    difference_vector =[]
    for i in range(len(true)):
        difference = true[i] - computed[i]
        difference_vector.append(difference)
    return difference_vector


def matrix_difference(true, computed):
    difference_matrix = []
    for i in range(len(true)):
        difference_row = []
        for j in range(len(true[i])):
            difference = true[i][j] - computed[i][j]
            difference_row.append(difference)
        difference_matrix.append(difference_row)
    return difference_matrix


def euclidean_norm(vector):
    sum_of_squares = sum(x**2 for x in vector)
    return sum_of_squares ** 0.5


def frobenius_norm(matrix):
    n = len(matrix)
    sum_of_squares = 0
    for i in range(n):
        for j in range(n):
            sum_of_squares = sum_of_squares + matrix[i][j] ** 2
    norm = sum_of_squares ** 0.5
    return norm


def calculate_mean(numbers):
    total_sum = sum(numbers)
    count = len(numbers)
    mean = total_sum / count
    return mean

#**Subroutine 1**


In [ ]:
def _2D_array_product(L, v):
    """
    Compute the product w = L * v given the vector v and the unit lower triangular matrix L.

    Parameters:
    L: Unit lower triangular matrix (2D array).
    v: Input vector (1D array)

    Returns:
    w: Resulting vector w = L * v (1D array)
    """

    # Initializations
    n = len(L)
    w = [0] * n

    for i in range(n):
        # Since L is unit lower triangular, L[i][i] = 1. Start with w[i] = v[i]
        w[i] = v[i]

        for j in range(i):
            w[i] = w[i] + L[i][j] * v[j]

    return w

#**Subroutine 1 Tester**

In [ ]:
def main():

    first_n = int(input("Enter the smallest size matrix to test: "))
    last_n = int(input("Enter the largest size matrix to test: "))
    trials = int(input("Enter the number of trials to run for each matrix size: "))
    low = int(input("Enter the lower bound for the integer entries: "))
    high = int(input("Enter the upper bound for the integer entries: "))

    error_mean_list = []
    for i in range(first_n, last_n+1):
        error_list = []
        for j in range(trials):
            matrix = generate_random_ULTM(i, low, high)
            vector = generate_random_vector(i, low, high)

            vector_1D = [item[0] for item in vector]

            pre_numpy_result = np.dot(np.array(matrix), np.array(vector_1D))
            numpy_result = pre_numpy_result.tolist()

            algorithm_result = _2D_array_product(matrix, vector_1D)

            difference_vector = vector_difference(numpy_result, algorithm_result)

            error = euclidean_norm(difference_vector)

            error_list.append(error)

        error_mean = calculate_mean(error_list)
        error_mean_list.append(error_mean)

    print(error_mean_list)

    trial_sizes = list(range(first_n, last_n+1))
    plt.scatter(trial_sizes, error_mean_list)
    plt.title("Mean Error per Dimension (Subroutine 1)")
    plt.xlabel("Dimension")
    plt.ylabel("Mean Error (Subroutine 1)")
    #plt.savefig("Routine1_1000to1010")
    plt.show


main()

#**Subroutine 2**

In [ ]:
def compressed_row_product(L, v):
    """
    Compute the product w = L * v given the vector v and the unit lower triangular matrix L.

    Parameters:
    L: Unit lower triangular matrix (1D array with compressed rows).
    v: Input vector (1D array)

    Returns:
    w: Resulting vector w = L * v (1D array)
    """

    # Initializations
    n = len(v)
    w = [0] * n

    L_index = 0  # Counts progress in L array
    for i in range(n):
        # Since L is unit lower triangular, L[i][i] = 1. Start with w[i] = v[i]
        w[i] = v[i]

        for j in range(i):
            w[i] = w[i] + L[L_index] * v[j]
            L_index = L_index + 1

    return w

#**Subroutine 2 Tester**

In [ ]:
def main():

    first_n = int(input("Enter the smallest size matrix to test: "))
    last_n = int(input("Enter the largest size matrix to test: "))
    trials = int(input("Enter the number of trials to run for each matrix size: "))
    low = int(input("Enter the lower bound for the integer entries: "))
    high = int(input("Enter the upper bound for the integer entries: "))

    error_mean_list = []
    for i in range(first_n, last_n+1):
        error_list = []
        for j in range(trials):
            matrix = generate_random_ULTM(i, low, high)
            vector = generate_random_vector(i, low, high)
            #print(matrix)
            #print(vector)

            # Create 1D array for algorithm to iterate through
            list_matrix = matrix.tolist()
            #print(list_matrix)

            CR_list = []
            for a in range(len(list_matrix)):
                for b in range(a):
                    CR_list.append(list_matrix[a][b])

            #print(CR_list)


            vector_1D = [item[0] for item in vector]

            pre_numpy_result = np.dot(np.array(matrix), np.array(vector_1D))
            #print(pre_numpy_result)

            numpy_result = pre_numpy_result.tolist()
            #print(numpy_result)

            algorithm_result = compressed_row_product(CR_list, vector_1D)
            #print(algorithm_result)

            difference_vector = vector_difference(numpy_result, algorithm_result)

            error = euclidean_norm(difference_vector)

            error_list.append(error)

        #print(error_list)
        error_mean = calculate_mean(error_list)
        error_mean_list.append(error_mean)

    print(error_mean_list)

    trial_sizes = list(range(first_n, last_n+1))
    plt.scatter(trial_sizes, error_mean_list)
    plt.title("Mean Error per Dimension (Subroutine 2)")
    plt.xlabel("Dimension")
    plt.ylabel("Mean Error (Subroutine 2)")
    #plt.savefig("Routine2_1000to1010")
    plt.show


main()

#**Subroutine 3**

In [ ]:
def banded_product(L, v):
    """
    Compute the product w = L * v given the vector v and the banded unit lower triangular matrix L.

    Parameters:
    L: Banded unit lower triangular matrix (1D array with compressed rows).
    v: Input vector (1D array)

    Returns:
    w: Resulting vector w = L * v (1D array)
    """

    # Initializations
    n = len(v)
    w = [0] * n

    # First and second entries are trivial
    w[0] = v[0]
    w[1] = v[1] + L[0] * v[0]

    L_index = 1  # Counts progress in L array
    for i in range(2,n):
        w[i] = v[i] + L[L_index] * v[i-2] + L[L_index+1] * v[i-1]
        L_index = L_index + 2

    return w

#**Subroutine 3 Tester**

In [ ]:
def main():

    first_n = int(input("Enter the smallest size matrix to test: "))
    last_n = int(input("Enter the largest size matrix to test: "))
    trials = int(input("Enter the number of trials to run for each matrix size: "))
    low = int(input("Enter the lower bound for the integer entries: "))
    high = int(input("Enter the upper bound for the integer entries: "))

    error_mean_list = []
    for i in range(first_n, last_n+1):
        error_list = []
        for j in range(trials):
            matrix = generate_random_BULTM(i, low, high)
            vector = generate_random_vector(i, low, high)
            #print(matrix)
            #print(vector)

            # Create 1D array for algorithm to iterate through
            list_matrix = matrix.tolist()
            #print(list_matrix)

            BCR_list = []
            BCR_list.append(list_matrix[1][0])
            for a in range(2,len(list_matrix)):
                for b in range(a-2,a):
                    BCR_list.append(list_matrix[a][b])

            #print(BCR_list)


            vector_1D = [item[0] for item in vector]

            pre_numpy_result = np.dot(np.array(matrix), np.array(vector_1D))
            #print(pre_numpy_result)

            numpy_result = pre_numpy_result.tolist()
            #print(numpy_result)

            algorithm_result = banded_product(BCR_list, vector_1D)
            #print(algorithm_result)

            difference_vector = vector_difference(numpy_result, algorithm_result)

            error = euclidean_norm(difference_vector)

            error_list.append(error)

        #print(error_list)
        error_mean = calculate_mean(error_list)
        error_mean_list.append(error_mean)

    print(error_mean_list)

    trial_sizes = list(range(first_n, last_n+1))
    plt.scatter(trial_sizes, error_mean_list)
    plt.title("Mean Error per Dimension (Subroutine 3)")
    plt.xlabel("Dimension")
    plt.ylabel("Mean Error (Subroutine 3)")
    #plt.savefig("Routine3_1000to1010")
    plt.show


main()

#**Subroutine 4**

In [ ]:
def compute_M_from_LU(LU):
    n = len(LU)
    M = [[0] * n for i in range(n)]

    # Perform matrix multiplication M = L * U
    for i in range(n):
        for j in range(n):
            if i == j:
                M[i][j] = LU[i][j]
                for k in range(i):
                    M[i][j] = M[i][j] + LU[i][k] * LU[k][i]
            elif i > j:
                for k in range(j+1):
                    M[i][j] = M[i][j] + LU[i][k] * LU[k][j]
            elif i < j:
                for k in range(i):
                    M[i][j] = M[i][j] + LU[i][k] * LU[k][j]
                M[i][j] = M[i][j] + 1 * LU [i][j]

    return M

#**Subroutine 4 Tester**

In [ ]:
def main():

    first_n = int(input("Enter the smallest size matrix to test: "))
    last_n = int(input("Enter the largest size matrix to test: "))
    trials = int(input("Enter the number of trials to run for each matrix size: "))
    low = int(input("Enter the lower bound for the integer entries: "))
    high = int(input("Enter the upper bound for the integer entries: "))

    error_mean_list = []
    for i in range(first_n, last_n+1):
        error_list = []
        for j in range(trials):
            L_matrix = generate_random_ULTM(i, low, high)
            U_matrix = generate_random_UTM(i, low, high)
            #print(L_matrix)
            #print(U_matrix)


            #vector_1D = [item[0] for item in vector]

            pre_numpy_result = np.dot(np.array(L_matrix), np.array(U_matrix))
            numpy_result = pre_numpy_result.tolist()
            #print(numpy_result)

            L_matrix_list = L_matrix.tolist()
            U_matrix_list = U_matrix.tolist()
            #print(L_matrix_list)
            #print(U_matrix_list)

            LU = construct_LU(L_matrix_list,U_matrix_list)
            #print(LU)
            algorithm_result = compute_M_from_LU(LU)
            #print(algorithm_result)

            difference_matrix = matrix_difference(numpy_result, algorithm_result)

            error = frobenius_norm(difference_matrix)

            error_list.append(error)

        #print(error_list)
        error_mean = calculate_mean(error_list)
        error_mean_list.append(error_mean)

    print(error_mean_list)

    trial_sizes = list(range(first_n, last_n+1))
    plt.scatter(trial_sizes, error_mean_list)
    plt.title("Mean Error per Dimension (Subroutine 4)")
    plt.xlabel("Dimension")
    plt.ylabel("Mean Error (Subroutine 4)")
    #plt.savefig("Routine4_1000to1010")
    plt.show


main()